In [ ]:
%pip install langchain openai beautifulsoup4 playwright faiss-cpu tiktoken python-dotenv nest_asyncio playwright langchain-community
import subprocess
subprocess.run(["playwright", "install"])

In [ ]:
from bs4 import BeautifulSoup
import nest_asyncio
import asyncio
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from dotenv import load_dotenv
import os
from urllib.parse import urljoin, urlparse
from playwright.async_api import async_playwright


In [ ]:
BASE_URL="https://victorsna.github.io"
START_PATH = "/wiki_ai/#/"

In [ ]:
# Carrega variáveis de ambiente do .env
load_dotenv()

# Garante que a chave da OpenAI está disponível
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("A chave OPENAI_API_KEY não foi encontrada no arquivo .env")

In [ ]:
nest_asyncio.apply()


In [ ]:
async def crawl_docs(base_url, start_path, max_pages=10):
    visited = set()
    to_visit = [start_path]
    documents = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        while to_visit and len(visited) < max_pages:
            path = to_visit.pop(0)
            if path in visited:
                continue

            url = urljoin(base_url, path)
            print(f"Crawling: {url}")
            try:
                await page.goto(url, timeout=60000)
                await page.wait_for_timeout(2000)
                text = await page.inner_text("body")

                documents.append(Document(page_content=text, metadata={"source": url}))

                # Coleta links internos da página já renderizada
                hrefs = await page.eval_on_selector_all("a", "elements => elements.map(el => el.href)")
                for href in hrefs:
                    parsed = urlparse(href)
                    
                    if href not in visited and href not in to_visit and parsed.query == "":
                        rel_path = parsed.path
                        fragment = parsed.fragment

                        to_visit.append(rel_path[:-1] + "#" + fragment)
            except Exception as e:
                print(f"Erro ao acessar {url}: {e}")

            visited.add(path)

        await browser.close()

    return documents

In [ ]:
raw_docs = await crawl_docs(BASE_URL, START_PATH, max_pages=15)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = splitter.split_documents(raw_docs)

In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name="gpt-3.5-turbo"),
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
query = "Quais headers são obrigatórios na API para adicionar o produto no carrinho?"
result = qa_chain(query)

print("\nResposta:\n", result["result"])